In [ ]:
#%run prelude.rc

import enum
import importlib.util
import sys
from pathlib import Path


import polars as pl
import numpy as np
import scipy.integrate as integrate
import HErmes as he
import HErmes.fitting as fit
import scipy.stats as st
import matplotlib

from scipy.spatial.transform import Rotation as rot
from datetime import datetime, UTC, timezone
from glob import glob


#pybindings
from pathlib import Path
import dashi as d
d.visual()
import tqdm


import matplotlib.pyplot as plt
import charmingbeauty as cb
lo = cb.layout
cb.visual.set_style_present()


import re
!export DJANGO_ALLOW_ASYNC_UNSAFE=1
import os
from matplotlib import font_manager
from matplotlib import rcParams


os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = '1'
plt.rcParams.update({'text.usetex' : False})


from matplotlib import font_manager


rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Open Sans']



In [ ]:
#!/usr/bin/env python


import gondola as gon
import time

files = gon.io.grace_get_telemetry_binaries(
    1765835400,
    1765935400, #end time 1767979800
    '/home/gaps/tof-data/antarctica/nextcloud/flight_2025-26'
)


lpt = None
toml_find = False


for f in files:  # in files from flight
    reader = gon.io.TelemetryPacketReader(f)

    for pack in reader:

        if pack.packet_type == gon.packets.TelemetryPacketType.AnyTofHK:
            tp = gon.packets.TofPacket.from_bytestream(pack.payload, 0)

            if tp.packet_type == gon.packets.TofPacketType.LiftofSettings:
                lpt = tp
                toml_find = True

        if toml_find is True:
            if (
                pack.packet_type == gon.packets.TelemetryPacketType.BoringEvent
                or pack.packet_type == gon.packets.TelemetryPacketType.InterestingEvent
            ):

                toml_find = False
                ev = gon.events.TelemetryEvent.from_telemetrypacket(pack)

                print("++++ new run ++++")
                print(f)
                print(ev.tof.run_id)
                # Decompress TOML
                gon.io.decompress_toml(lpt.payload, 'test.toml')
                # Open and print first 130 lines
                with open("test.toml", "r") as toml_file:
                    print("run ID:" + str(ev.tof.run_id))
                    print("timestamp: " + str(int(np.round(ev.tof.timestamp48/(10**8)))))
                    for i, line in enumerate(toml_file):
                        if i >= 30:
                            break
                        print(line.rstrip())
                print("---------------------------------------")

                # Now process event
                


In [ ]:
import time
import gondola as gon

with open("tof_flight_runs_new.txt", "w") as outfile:

    #outfile.write(f"Script start time (unix): {time.time()}\n\n")

    files = gon.io.grace_get_telemetry_binaries(
        1765835400,
        1767979800,  # end time
        '/home/gaps/tof-data/antarctica/nextcloud/flight_2025-26'
    )
    
    lpt = None
    toml_find = False

    for f in files:  # in files from flight
        reader = gon.io.TelemetryPacketReader(f)

        for pack in reader:

            if pack.packet_type == gon.packets.TelemetryPacketType.AnyTofHK:
                tp = gon.packets.TofPacket.from_bytestream(pack.payload, 0)

                if tp.packet_type == gon.packets.TofPacketType.LiftofSettings:
                    lpt = tp
                    toml_find = True
            
            if toml_find is True:
                if (
                    pack.packet_type == gon.packets.TelemetryPacketType.BoringEvent
                    or pack.packet_type == gon.packets.TelemetryPacketType.InterestingEvent
                ):
                    toml_find = False
                    ev = gon.events.TelemetryEvent.from_telemetrypacket(pack)
                    
                    outfile.write("++++ new run ++++\n")
                    outfile.write(f"File: {f}\n")
                    outfile.write(f"Run ID: {ev.tof.run_id}\n\n")
                    outfile.write("timestamp start: " + str(int(np.round(ev.tof.timestamp48/(10**8)))) + "\n\n")
                    # Decompress TOML
                    gon.io.decompress_toml(lpt.payload, 'test.toml')
                    # Open and print first 130 lines
                    outfile.write("\n ----- toml  -----\n")
                    with open("test.toml", "r") as toml_file:
                        for i, line in enumerate(toml_file):
                            if i >= 130:
                                break
                            outfile.write(line.rstrip() + "\n")
                    outfile.write("\n---------------------------------------\n")